In [ ]:
!pip install --no-compile genQC==0.2.5 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.2 MB/s eta 0:00:00


In [ ]:
!pip install cudaq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.9/128.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.9/112.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.1/277.1 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 4.8 MB/s eta 0:00:00
  Created wheel for cudaq: filename=cudaq-0.14.0-py3-none-any.whl size=7829 sha256=fba8839d93e580547ee5c271deb9d3706aa33d7882500a8314bc3f3a4df34bfc
  Stored in directory: /root/.cache/pip/wheels/d1/6f/69/6c1c103b341ba72132776a334aae3a495fe8430de1fe00bb3e
Successfully built cudaq
  Attempting uninstall: cupy-cud

In [ ]:
import cudaq
import torch
import numpy as np
import genQC
import os

os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'

# Users may hit a rate limit for HuggingFace requests if the HF_TOKEN is not set.
import huggingface_hub
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    huggingface_hub.login(token=hf_token, add_to_git_credential=False)

import genQC.utils.misc_utils as util
from genQC.pipeline.diffusion_pipeline import DiffusionPipeline
from genQC.pipeline.multimodal_diffusion_pipeline \
            import MultimodalDiffusionPipeline_ParametrizedCompilation

from genQC.platform.tokenizer.circuits_tokenizer import CircuitTokenizer
from genQC.platform.simulation import Simulator, CircuitBackendType
from genQC.scheduler.scheduler_dpm import DPMScheduler

from genQC.inference.sampling \
            import decode_tensors_to_backend, generate_compilation_tensors
from genQC.inference.evaluation_helper import get_unitaries
from genQC.inference.eval_metrics import UnitaryInfidelityNorm

In [ ]:
 # Use CUDA if we have a GPU available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
# Flag to only run large model if GPU available
RUN_LARGE_MODEL = ( device == torch.device("cuda") )
print(RUN_LARGE_MODEL)

True


In [ ]:
util.set_seed(0)

In [ ]:
simulator = Simulator(CircuitBackendType.CUDAQ,
                      target='qpp-cpu')

In [ ]:
discrete_pipeline = DiffusionPipeline.from_pretrained(
            repo_id="Floki00/qc_unitary_3qubit", # Download model from Hugging Face
            device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[INFO]: `genQC.models.unet_qc.QC_Compilation_UNet` instantiated from given `config` on cuda.
[INFO]: Loading model from `/root/.cache/huggingface/hub/models--Floki00--qc_unitary_3qubit/snapshots/e12810ea65a9a88832fadd34834894e8174658a5/model.pt` onto device: cuda.
[INFO]: `genQC.models.unet_qc.QC_Compilation_UNet`. Freeze model: True
[INFO]: `genQC.models.frozen_open_clip.CachedFrozenOpenCLIPEmbedder` instantiated from given `config` on cuda.
[INFO]: `genQC.models.frozen_open_clip.CachedFrozenOpenCLIPEmbedder`. Found no key `save_type` in `config`. No state dict loaded.
[INFO]: `genQC.models.frozen_open_clip.CachedFrozenOpenCLIPEmbedder`. Freeze model: True


In [ ]:
cont_pipeline = MultimodalDiffusionPipeline_ParametrizedCompilation.from_pretrained(
    repo_id="Floki00/cirdit_multimodal_compile_3to5qubit_v1.1", # Download model from Hugging Face
    device=device)

[INFO]: Loading tensor from `/root/.cache/huggingface/hub/models--Floki00--cirdit_multimodal_compile_3to5qubit_v1.1/snapshots/bd2c968349d0915580b5acbaf970d2fc446d2503/beta_schedule_learned_linear.safetensors` onto device: cuda.
[INFO]: Loaded beta_schedule (path:/root/.cache/huggingface/hub/models--Floki00--cirdit_multimodal_compile_3to5qubit_v1.1/snapshots/bd2c968349d0915580b5acbaf970d2fc446d2503/beta_schedule_learned_linear.safetensors).
[INFO]: Creating `UnitaryTextEncoder` with `unitary_downscale_factor=4` and `encoding_ch=640`.
[INFO]: `genQC.models.transformers.cirdit_multimodal.UnitaryCLIPPartialNoiseCompilationCirDiT` instantiated from given `config` on cuda.
[INFO]: Loading model from `/root/.cache/huggingface/hub/models--Floki00--cirdit_multimodal_compile_3to5qubit_v1.1/snapshots/bd2c968349d0915580b5acbaf970d2fc446d2503/model.safetensors` onto device: cuda.
[INFO]: `genQC.models.transformers.cirdit_multimodal.UnitaryCLIPPartialNoiseCompilationCirDiT`. Freeze model: True
[INFO

In [ ]:
cont_pipeline.gate_pool

['h', 'cx', 'ccx', 'swap', 'rx', 'ry', 'rz', 'cp']

In [ ]:
cont_vocabulary = {g:i+1 for i, g in enumerate(cont_pipeline.gate_pool)}
cont_tokenizer  = CircuitTokenizer(cont_vocabulary)
cont_tokenizer.vocabulary

{'h': 1, 'cx': 2, 'ccx': 3, 'swap': 4, 'rx': 5, 'ry': 6, 'rz': 7, 'cp': 8}

In [ ]:
# These parameters are specific to our pre-trained model.
cont_system_size   = 5
cont_max_gates     = 32

In [ ]:
cont_pipeline.scheduler   = DPMScheduler.from_scheduler(cont_pipeline.scheduler)
cont_pipeline.scheduler_w = DPMScheduler.from_scheduler(cont_pipeline.scheduler_w)

timesteps = 40
cont_pipeline.scheduler.set_timesteps(timesteps)
cont_pipeline.scheduler_w.set_timesteps(timesteps)

cont_pipeline.lambda_h = 1.0
cont_pipeline.lambda_w = 0.35
cont_pipeline.g_h = 0.3
cont_pipeline.g_w = 0.1

[INFO]: Loading tensor from `/root/.cache/huggingface/hub/models--Floki00--cirdit_multimodal_compile_3to5qubit_v1.1/snapshots/bd2c968349d0915580b5acbaf970d2fc446d2503/beta_schedule_learned_linear.safetensors` onto device: cuda.
[INFO]: Loaded beta_schedule (path:/root/.cache/huggingface/hub/models--Floki00--cirdit_multimodal_compile_3to5qubit_v1.1/snapshots/bd2c968349d0915580b5acbaf970d2fc446d2503/beta_schedule_learned_linear.safetensors).


In [ ]:
def verify_unitary(U: torch.Tensor):
    """Check if unitary."""
    assert torch.allclose(U.adjoint() @ U, torch.eye(2**num_of_qubits, dtype=U.dtype))
    assert torch.allclose(U @ U.adjoint(), torch.eye(2**num_of_qubits, dtype=U.dtype))

In [ ]:
def sample_kernels_and_evaluate(U: torch.Tensor,
                                prompt: str,
                                num_of_qubits: int,
                                samples: int,
                                discrete_model: bool,
                                return_tensors: bool = False):
    """
    Sample the DM and return generated kernels with coresponding infidelities.
    """

    # 1) Check if unitary
    verify_unitary(U)

    # 2) Generate tensor representations using the DM based on the prompt and U.
    U = U.to(torch.complex64)

    if discrete_model:
        # Sample discrete model
        out_tensor = generate_compilation_tensors(discrete_pipeline,
                                  prompt=prompt,
                                  U=U,
                                  samples=samples,      # How many circuits we sample per unitary
                                  system_size=discrete_system_size,
                                  num_of_qubits=num_of_qubits,
                                  max_gates=discrete_max_gates,
                                  g=10.0,               # classifier-free-guidance (CFG) scale
                                  no_bar=True,          # show progress bar
                                  auto_batch_size=256,  # for less GPU memory usage limit batch size
                                  tensor_prod_pad=False,
                                  enable_params=False,
                                 )
        tokenizer = discrete_tokenizer
        params    = None

    else:
        if not RUN_LARGE_MODEL:
            print(f">> Skipped sampling large model. Flag: {RUN_LARGE_MODEL=} <<")
            if return_tensors:
                return [], [], []
            return [], []

        # Sample continuous model
        out_tensor, params = generate_compilation_tensors(cont_pipeline,
                                  prompt=prompt,
                                  U=U,
                                  samples=samples,     # How many circuits we sample per unitary
                                  system_size=cont_system_size,
                                  num_of_qubits=num_of_qubits,
                                  max_gates=cont_max_gates,
                                  no_bar=True,         # show progress bar
                                  auto_batch_size=256, # for less GPU memory usage limit batch size
                                 )
        tokenizer = cont_tokenizer

    # 3) Convert tensors to kernels
    generated_kernels, _, generated_tensors = decode_tensors_to_backend(simulator=simulator,
                                                     tokenizer=tokenizer,
                                                     tensors=out_tensor,
                                                     params=params,
                                                     return_tensors=True)

    # 4) Evaluate the kernels and return the unitaries
    generated_us = get_unitaries(simulator, generated_kernels, num_qubits=num_of_qubits)

    # 5) Calculate the infidelities to the target U
    infidelities = UnitaryInfidelityNorm.distance(
                    approx_U=torch.from_numpy(np.stack(generated_us)).to(torch.complex128),
                    target_U=U.unsqueeze(0).to(torch.complex128))

    if return_tensors:
        return generated_kernels, infidelities, generated_tensors
    return generated_kernels, infidelities

In [ ]:
def plot_topk_kernels(generated_kernels: list,
                      infidelities: torch.Tensor,
                      num_of_qubits:int,
                      topk: int):
    """
    Plot the topk best generated kernels.
    """

    # Get topk indices
    best_indices = np.argsort(infidelities)[:topk]

    input_state = [0] * (2**num_of_qubits)
    input_state[0] = 1

    # Print the circuits
    for i, best_index in enumerate(best_indices):
        kernel = generated_kernels[best_index].kernel
        thetas = generated_kernels[best_index].params

        print(f"Circuit has an infidelity of {infidelities[best_index].item():0.1e}.")
        print(cudaq.draw(kernel, input_state, thetas))